In [1]:
import pandas as pd
import glob
import os
import re

In [2]:
### only files starting with stats

files = glob.glob('G:/My Drive/GitHubProjects/MLS/data/raw/matches/teams/stats*.csv')


In [3]:
df = [] 

for file in files:
    df.append(pd.read_csv(file))
df = pd.concat(df, ignore_index=True)
df.to_csv('G:/My Drive/GitHubProjects/MLS/data/interim/matches/teams/mls_match_team_stats_0326.csv', index=False)

In [4]:
df

,stat_name,home_value,away_value,category,tip_id,home_possession,home_advantage,away_possession,away_advantage,match_id,date
0,Possession %,58.3,41.7,general,NaN,NaN,NaN,NaN,NaN,763ad58d,Saturday March 30 2024
1,Shots,12.0,17.0,general,NaN,NaN,NaN,NaN,NaN,763ad58d,Saturday March 30 2024
2,Shots on Goal,2.0,10.0,general,NaN,NaN,NaN,NaN,NaN,763ad58d,Saturday March 30 2024
3,Blocked Shots,4.0,4.0,general,NaN,NaN,NaN,NaN,NaN,763ad58d,Saturday March 30 2024
4,Total Passes,606.0,370.0,general,NaN,NaN,NaN,NaN,NaN,763ad58d,Saturday March 30 2024
...,...,...,...,...,...,...,...,...,...,...,...
60561,NaN,NaN,NaN,possession,bar_2_7,Home Possession: 45.45%,Home Advantage: N/A,Away Possession: 54.55%,Away Advantage: 9.1%,7630999999999999289751995875466473865476733141...,Sunday February 22\nLumen Field
60562,NaN,NaN,NaN,possession,bar_2_8,Home Possession: 29.17%,Home Advantage: N/A,Away Possession: 70.83%,Away Advantage: 41.66%,7630999999999999289751995875466473865476733141...,Sunday February 22\nLumen Field
60563,Total Team XG,1.4,0.6,xg,NaN,NaN,NaN,NaN,NaN,7630999999999999289751995875466473865476733141...,Sunday February 22\nLumen Field
60564,Shots,14.0,7.0,xg,NaN,NaN,NaN,NaN,NaN,7630999999999999289751995875466473865476733141...,Sunday February 22\nLumen Field


In [ ]:
##combine first and second column into one column

df_cleaned['stat'] = df['category'].astype(str) + '_' + df['stat_name'].astype(str)

In [ ]:
df = df_cleaned.drop(columns=['category', 'stat_name'])

In [ ]:
df

In [5]:
## rename bars to every 5 minutes

bar_dict = {
    '0-5': 'bar_0',
    '6-10': 'bar_1',
    '11-15': 'bar_2',
    '16-20': 'bar_3',
    '21-25': 'bar_4',
    '26-30': 'bar_5',
    '31-35': 'bar_6',
    '36-40': 'bar_7',
    '41-45': 'bar_8',
    '46-50': 'bar_2_0',
    '51-55': 'bar_2_1',
    '56-60': 'bar_2_2',
    '61-65': 'bar_2_3',
    '66-70': 'bar_2_4',
    '71-75': 'bar_2_5',
    '76-80': 'bar_2_6',
    '81-85': 'bar_2_7',
    '86-90': 'bar_2_8',
}

bar_dict_switched = {v: k for k, v in bar_dict.items()}

df['tip_id'] = df['tip_id'].astype(str).str.strip() 

df['tip_id'] = df['tip_id'].replace(bar_dict_switched)

In [ ]:
df

In [10]:
mask = df["tip_id"].astype(str).str.match(r"^\d{1,2}-\d{1,2}$", na=False)

h_pct = df.loc[mask, "home_possession"].astype(str).str.extract(r"(\d+(?:\.\d+)?)")[0].astype(float)
a_pct = df.loc[mask, "away_possession"].astype(str).str.extract(r"(\d+(?:\.\d+)?)")[0].astype(float)

df.loc[mask, "home_value"] = h_pct.values
df.loc[mask, "away_value"] = a_pct.values

df.loc[mask, "stat"] = "possession_" + df.loc[mask, "tip_id"].str.replace("-", "_", regex=False)

In [ ]:
df  = df.drop(columns=['tip_id', 'home_possession', 'away_possession'])
df

In [ ]:
### reset column order

df = df[['match_id', 'date', 'stat', 'home_value', 'away_value', 'home_team', 'away_team']]

In [ ]:
df

In [ ]:
df.to_csv('G:/My Drive/GitHubProjects/MLS/data_files/cleaned/matches/teams/team_stats_cleaned.csv', index=False)

In [6]:
def extract_team_and_date(filename):
    base = os.path.basename(filename)
    name, ext = os.path.splitext(base)
    name = name.replace('stats_', '')
    teams = name.split('-')[0]
    date = name.split('-')[1:4]
    home_team = teams.split('vs')[0]
    away_team = teams.split('vs')[1]
    date = '-'.join(date)
    return home_team, away_team, date

In [7]:
def clean_teams(df):
    df = df.copy()
    df = df.drop(columns=['home_advantage', 'away_advantage'])
    df['stat'] = df['category'].astype(str) + '_' + df['stat_name'].astype(str)
    df = df.drop(columns=['category', 'stat_name'])
    df['tip_id'] = df['tip_id'].astype(str).str.strip() 
    df['tip_id'] = df['tip_id'].replace(bar_dict_switched)
    mask = df["tip_id"].astype(str).str.match(r"^\d{1,2}-\d{1,2}$", na=False)
    h_pct = df.loc[mask, "home_possession"].astype(str).str.extract(r"(\d+(?:\.\d+)?)")[0].astype(float)
    a_pct = df.loc[mask, "away_possession"].astype(str).str.extract(r"(\d+(?:\.\d+)?)")[0].astype(float)
    df.loc[mask, "home_value"] = h_pct.values 
    df.loc[mask, "away_value"] = a_pct.values
    df.loc[mask, "stat"] = "possession_" + df.loc[mask, "tip_id"].str.replace("-", "_", regex=False)
    df = df.drop(columns=['tip_id', 'home_possession', 'away_possession'])
    df = df[['stat', 'home_value', 'away_value', 'match_id']]
    return df

In [8]:
df2 = clean_teams(df)

In [9]:
df2

,stat,home_value,away_value,match_id
0,general_Possession %,58.30,41.70,763ad58d
1,general_Shots,12.00,17.00,763ad58d
2,general_Shots on Goal,2.00,10.00,763ad58d
3,general_Blocked Shots,4.00,4.00,763ad58d
4,general_Total Passes,606.00,370.00,763ad58d
...,...,...,...,...
60561,possession_81_85,45.45,54.55,7630999999999999289751995875466473865476733141...
60562,possession_86_90,29.17,70.83,7630999999999999289751995875466473865476733141...
60563,xg_Total Team XG,1.40,0.60,7630999999999999289751995875466473865476733141...
60564,xg_Shots,14.00,7.00,7630999999999999289751995875466473865476733141...


In [10]:
df2.to_csv('G:/My Drive/GitHubProjects/MLS/data/interim/matches/teams/team_stats_cleaned_v2_0326.csv', index=False)